# Delhi Colonies Public Services Index (updated 2025)

## Compute the following indices:
* Index with bounding box neighbors [effective service count divided by population]
* Index with bounding box neighbors [effective service count divided by population/area]

### How to compute indices
* Load in colonies dataset (bounding box only) from joblib **[done]**
* Merge New Population Estimates **[done]**
* Remove Rural Villages **[done]**
* Import services shapefiles **[done]**
    * Make sure correct file paths exist
    * Ensure that all shapefiles are valid using `check_shapefile` function
    * Reproject shapefiles to EPSG 7760 (if needed)
* Compute all Services indices (turn into a function) **[done]**
    * bbox neighbors, Population Size
    * bbox neighbors, Population Density

## Import modules and set constants

In [1]:
import os
import joblib
from importlib import reload
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, box
import spatial_index_utils
from spatial_index_utils import calc_all_services

In [2]:
#reload(spatial_index_utils)

## Constants

In [22]:
# WGS 84 / Delhi
epsg_code = 7760

data_dir = "/home/bwbelljr/delhi_data/"

colonies_bbox_file = os.path.join(data_dir, 'colonies_bbox_nbrs2025.joblib')

popfile_2020 = os.path.join(data_dir, "pop_colony_wp_2020_jjc_adjusted.csv")
for popfile in [popfile_2020]:
    print(f"{popfile} exists: {os.path.exists(popfile)}")

# Define filepaths
services_dir = os.path.join(data_dir, 'Public Services')

bank_fp = os.path.join(services_dir, 'Banking', 'Banking.shp')
health_fp = os.path.join(services_dir, 'Health', 'Health.shp')
road_fp = os.path.join(services_dir, 'Major Road', 'Road.shp')
police_fp = os.path.join(services_dir, 'Police', 'Police Station.shp')
ration_fp = os.path.join(services_dir, 'Ration', 'Ration.shp')
school_fp = os.path.join(services_dir, 'School', 'schools7760.shp')
transport_fp = os.path.join(services_dir, 'Transport', 'Transport.shp')

missing_colonies_csv_path = os.path.join(data_dir, "missing_colonies.csv")

# boundary of Delhi
delhi_bounds_filepath = os.path.join(data_dir, 'delhi_bounds_buffer', 'delhi_bounds_buffer.shp')

# Check that all filepaths exist
filepath_list = [bank_fp, health_fp, road_fp, police_fp, ration_fp, school_fp, transport_fp, delhi_bounds_filepath]

for filepath in filepath_list:
    if not os.path.exists(filepath):
        print('{} does not exist'.format(filepath))

psi_results_dir = os.path.join(data_dir, 'psi_2020_results')
colonies_bbox_psi_csv_file = os.path.join(psi_results_dir, 'delhi_psi_bbox_popsize2020_norv_12Sep2021.csv')
colonies_bbox_psi_joblib_file = os.path.join(psi_results_dir, 'colonies_bbox_psi_popsize2020_norv_12Sep2021.joblib')

colonies_bbox_psi_popdensity_csv_file = os.path.join(psi_results_dir, 'delhi_psi_bbox_popdensity2020_norv_12Sep2021.csv') 
colonies_bbox_psi_popdensity_joblib_file = os.path.join(psi_results_dir, 'colonies_bbox_psi_popdensity2020_norv_12Sep2021.joblib')

colonies_bbox_psi_popsize_file = os.path.join(psi_results_dir, 'delhi_psi_bbox_popsize2020_norv_12Sep2021.shp')
colonies_bbox_psi_popdensity_file = os.path.join(psi_results_dir, 'delhi_psi_bbox_popdensity2020_norv_12Sep2021.shp')

/home/bwbelljr/delhi_data/pop_colony_wp_2020_jjc_adjusted.csv exists: True


## Data Loading

In [4]:
with open(colonies_bbox_file, 'rb') as f:
    colonies_bbox_nbrs = joblib.load(f)

updated_pop = pd.read_csv(popfile_2020)

# Import services
bank = gpd.read_file(bank_fp)
health = gpd.read_file(health_fp)
road = gpd.read_file(road_fp)
police = gpd.read_file(police_fp)
ration = gpd.read_file(ration_fp)
school = gpd.read_file(school_fp)
transport = gpd.read_file(transport_fp)

## View Colonies Dataset (bounding box only)

In [5]:
colonies_bbox_nbrs.head()

,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,area_km2,canal,railway,drain,barrier,centroid,ndmc_dist_km,nbrs_bbox,nbrs_dist_bbox,index
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",1.966739,False,True,False,True,POINT (1020123.175 995898.851),5.159809,"[3508, 5598, 5599, 3776, 5602, 4011, 3491, 560...","[(3508, 0.8833017034389078), (5598, 1.07479036...",0
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",0.036429,False,False,False,False,POINT (1019673.024 994869.699),6.273149,"[5594, 5586]","[(5594, 0.6299162683013301), (5586, 0.35712228...",1
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",0.230739,False,False,False,False,POINT (1019485.484 994565.783),6.618792,"[5596, 5594, 5587, 5585]","[(5596, 0.679329044817247), (5594, 0.453264599...",2
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",0.281195,False,False,False,False,POINT (1019171.868 994576.688),6.709542,"[5596, 5586, 5588]","[(5596, 0.5564745103110775), (5586, 0.31380549...",3
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",0.301253,False,False,False,False,POINT (1018785.675 994590.275),6.839299,"[5596, 5587, 5621, 5620]","[(5596, 0.6272725729096971), (5587, 0.38643264...",4


In [6]:
len(colonies_bbox_nbrs)

4357

## Update/Merge Population Estimates

In [7]:
updated_pop.head()

,population,area,uso_area_u,uso_final
0,3570.061035,NEW DELHI 36,5584,Planned
1,320.568634,NEW DELHI 35,5585,Planned
2,2215.206543,NEW DELHI 34,5586,Planned
3,3956.166992,NEW DELHI 33,5587,Planned
4,3961.943359,NEW DELHI 32,5588,Planned


In [8]:
# rename population column to make distinct in upcoming merge
updated_pop = updated_pop.rename(columns={"population":"population_new"})
updated_pop.head()

,population_new,area,uso_area_u,uso_final
0,3570.061035,NEW DELHI 36,5584,Planned
1,320.568634,NEW DELHI 35,5585,Planned
2,2215.206543,NEW DELHI 34,5586,Planned
3,3956.166992,NEW DELHI 33,5587,Planned
4,3961.943359,NEW DELHI 32,5588,Planned


In [9]:
# Restrict dataframe to only two columns:
# layer: population data
# uso_area_u: unique id for colonies
updated_pop = updated_pop[['population_new', 'uso_area_u']]
updated_pop.head()

,population_new,uso_area_u
0,3570.061035,5584
1,320.568634,5585
2,2215.206543,5586
3,3956.166992,5587
4,3961.943359,5588


In [10]:
# Left merge updated population data with colonies data
colonies_bbox_nbrs = colonies_bbox_nbrs.merge(updated_pop, how='left', 
                          left_on="USO_AREA_U", right_on='uso_area_u')
colonies_bbox_nbrs.head()

,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,area_km2,canal,railway,drain,barrier,centroid,ndmc_dist_km,nbrs_bbox,nbrs_dist_bbox,index,population_new,uso_area_u
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",1.966739,False,True,False,True,POINT (1020123.175 995898.851),5.159809,"[3508, 5598, 5599, 3776, 5602, 4011, 3491, 560...","[(3508, 0.8833017034389078), (5598, 1.07479036...",0,3570.061035,5584.0
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",0.036429,False,False,False,False,POINT (1019673.024 994869.699),6.273149,"[5594, 5586]","[(5594, 0.6299162683013301), (5586, 0.35712228...",1,320.568634,5585.0
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",0.230739,False,False,False,False,POINT (1019485.484 994565.783),6.618792,"[5596, 5594, 5587, 5585]","[(5596, 0.679329044817247), (5594, 0.453264599...",2,2215.206543,5586.0
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",0.281195,False,False,False,False,POINT (1019171.868 994576.688),6.709542,"[5596, 5586, 5588]","[(5596, 0.5564745103110775), (5586, 0.31380549...",3,3956.166992,5587.0
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",0.301253,False,False,False,False,POINT (1018785.675 994590.275),6.839299,"[5596, 5587, 5621, 5620]","[(5596, 0.6272725729096971), (5587, 0.38643264...",4,3961.943359,5588.0


In [24]:
colonies_with_missing_population = colonies_bbox_nbrs[colonies_bbox_nbrs['population_new'].isna()]
colonies_with_missing_population.to_csv(missing_colonies_csv_path, index=False)
colonies_bbox_nbrs.drop(index=colonies_with_missing_population.index, inplace=True)

In [25]:
# Remove extraneous columns
colonies_bbox_nbrs = colonies_bbox_nbrs.drop(columns=['uso_area_u'])

# Rename 'population_new' column as 'population'
colonies_bbox_nbrs = colonies_bbox_nbrs.rename(columns={'population_new': 'population'})

colonies_bbox_nbrs.head()

,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,area_km2,canal,railway,drain,barrier,centroid,ndmc_dist_km,nbrs_bbox,nbrs_dist_bbox,index,population
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",1.966739,False,True,False,True,POINT (1020123.175 995898.851),5.159809,"[3508, 5598, 5599, 3776, 5602, 4011, 3491, 560...","[(3508, 0.8833017034389078), (5598, 1.07479036...",0,3570.061035
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",0.036429,False,False,False,False,POINT (1019673.024 994869.699),6.273149,"[5594, 5586]","[(5594, 0.6299162683013301), (5586, 0.35712228...",1,320.568634
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",0.230739,False,False,False,False,POINT (1019485.484 994565.783),6.618792,"[5596, 5594, 5587, 5585]","[(5596, 0.679329044817247), (5594, 0.453264599...",2,2215.206543
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",0.281195,False,False,False,False,POINT (1019171.868 994576.688),6.709542,"[5596, 5586, 5588]","[(5596, 0.5564745103110775), (5586, 0.31380549...",3,3956.166992
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",0.301253,False,False,False,False,POINT (1018785.675 994590.275),6.839299,"[5596, 5587, 5621, 5620]","[(5596, 0.6272725729096971), (5587, 0.38643264...",4,3961.943359


In [26]:
# Check that we have same number of colonies
len(colonies_bbox_nbrs)

4342

In [27]:
# Confirm no population estimates are missing
sum(colonies_bbox_nbrs['population'].isna())

0

## Remove "Rural Villages"

In [28]:
colonies_bbox_nbrs = colonies_bbox_nbrs[colonies_bbox_nbrs['USO_FINAL'] != 'RV']

In [29]:
# Check that number of colonies
len(colonies_bbox_nbrs)

4131

## Check validity of services shapefiles
* Duplicate rows are okay for ATMs (I assume that ATM locations for the same bank in a similar location will seem to be counted twice)
* Look specifically for invalid geometries and whether shapefile is fully contained within Delhi

In [30]:
spatial_index_utils.check_shapefile(gdf=bank, gdf_name='bank', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

bank has duplicate rows: True
----------------------------------------------------
rows with invalid geometries 

----------------------------------------------------
all geometries in bank are of type Point: True
----------------------------------------------------
Rows with None value in geometry column are below
Empty GeoDataFrame
Columns: [bank_name, Latitude, Longitude, Type, geometry, geom_type]
Index: []
----------------------------------------------------
bank shapefile is contained within Delhi: True
----------------------------------------------------
Done with shapefile evaluation


In [31]:
bank.shape

(10637, 6)

In [32]:
# Remove duplicate rows found in bank DataFrame
bank.drop_duplicates(inplace=True)

In [33]:
bank.shape

(9397, 6)

In [34]:
spatial_index_utils.check_shapefile(gdf=health, gdf_name='health', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

health has duplicate rows: False
----------------------------------------------------
rows with invalid geometries 

----------------------------------------------------
all geometries in health are of type Point: True
----------------------------------------------------
Rows with None value in geometry column are below
Empty GeoDataFrame
Columns: [Hospital_C, ADDRESS, X, Y, geometry, geom_type]
Index: []
----------------------------------------------------
health shapefile is contained within Delhi: True
----------------------------------------------------
Done with shapefile evaluation


In [35]:
spatial_index_utils.check_shapefile(gdf=road, gdf_name='road', 
                                    geom_type='Line', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

road has duplicate rows: False
----------------------------------------------------
rows with invalid geometries 

----------------------------------------------------
all geometries in road are of type Line: True
----------------------------------------------------
Rows with None value in geometry column are below
Empty GeoDataFrame
Columns: [FID, RD_NM, RD_CLS, RD_LANES, RD_TP_SRF, RD_MB, RD_ONEWAY, EL_GND, DIST_NM, ONEWAY, Speed_kmph, geometry, geom_type]
Index: []
----------------------------------------------------
road shapefile is contained within Delhi: True
----------------------------------------------------
Done with shapefile evaluation


In [36]:
spatial_index_utils.check_shapefile(gdf=police, gdf_name='police', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

police has duplicate rows: False
----------------------------------------------------
rows with invalid geometries 

----------------------------------------------------
all geometries in police are of type Point: True
----------------------------------------------------
Rows with None value in geometry column are below
Empty GeoDataFrame
Columns: [NAME, POLICE_STA, DISTRICT, x, y, geometry, geom_type]
Index: []
----------------------------------------------------
police shapefile is contained within Delhi: True
----------------------------------------------------
Done with shapefile evaluation


In [37]:
spatial_index_utils.check_shapefile(gdf=ration, gdf_name='ration', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

ration has duplicate rows: False
----------------------------------------------------
rows with invalid geometries 

----------------------------------------------------
all geometries in ration are of type Point: True
----------------------------------------------------
Rows with None value in geometry column are below
Empty GeoDataFrame
Columns: [S No., License No, FPS ID, Circle, FPS Shop N, Address Of, Latitude, Longitude, Source, geometry, geom_type]
Index: []
----------------------------------------------------
ration shapefile is contained within Delhi: True
----------------------------------------------------
Done with shapefile evaluation


In [38]:
spatial_index_utils.check_shapefile(gdf=school, gdf_name='school', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

school has duplicate rows: False
----------------------------------------------------
rows with invalid geometries 

----------------------------------------------------
all geometries in school are of type Point: True
----------------------------------------------------
Rows with None value in geometry column are below
Empty GeoDataFrame
Columns: [objectid_1, objectid, vilname, schname, schcd, schcat, school_cat, pincode, rururb, location, schtype, school_typ, schmgt, management, dtname, stname, stcode11, dtcode11, sdtcode11, sdtname, geometry, geom_type]
Index: []

[0 rows x 22 columns]
----------------------------------------------------
school shapefile is contained within Delhi: True
----------------------------------------------------
Done with shapefile evaluation


In [39]:
spatial_index_utils.check_shapefile(gdf=transport, gdf_name='transport', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

transport has duplicate rows: False
----------------------------------------------------
rows with invalid geometries 

----------------------------------------------------
all geometries in transport are of type Point: True
----------------------------------------------------
Rows with None value in geometry column are below
Empty GeoDataFrame
Columns: [stop_id, stop_name, stop_lat, stop_lon, Type, geometry, geom_type]
Index: []
----------------------------------------------------
transport shapefile is contained within Delhi: True
----------------------------------------------------
Done with shapefile evaluation


## Check CRS (all shapefiles should be in EPSG: 7760)

In [40]:
bank.crs == health.crs == road.crs == police.crs == ration.crs == school.crs == transport.crs

True

In [41]:
bank.crs

<Projected CRS: EPSG:7760>
Name: WGS 84 / Delhi
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: India - Delhi national capital territory.
- bounds: (76.83, 28.4, 77.34, 28.89)
Coordinate Operation:
- name: Delhi NSF LCC
- method: Lambert Conic Conformal (2SP)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [42]:
colonies_bbox_nbrs.crs == bank.crs

True

## Define Point and Line Services

In [43]:
# Define all point services as dictionary
# makes it easier to calculate all point
# services with one function
point_services = {'bank': bank,
                  'health': health,
                  'police': police,
                  'ration': ration,
                  'school': school,
                  'transport': transport}

line_services = {'road': road}

## Calculate all service indices in one function

### Calculate PSI for bbox neighbors using Population Size

In [44]:
colonies_bbox_psi_popsize = calc_all_services(polygon_gdf = colonies_bbox_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       pcen_denom = "pop",
                                       nbr_dist_colname = 'nbrs_dist_bbox')

colonies_bbox_psi_popsize = colonies_bbox_psi_popsize.rename(columns={'road_count':'road_length'})

GeoDataFrame now has the following CRS:

EPSG:7760


/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1129: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.011467577284127922' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, pcen_mobile_colname] = poly_count/denom
/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1212: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.018837853483668447' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, service_idx_colname] = result


bank service index is completed
--------------------------------------------------------
GeoDataFrame now has the following CRS:

EPSG:7760


/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1129: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.00018438975101874047' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, pcen_mobile_colname] = poly_count/denom
/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1212: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.0019613557082333744' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, service_idx_colname] = result


health service index is completed
--------------------------------------------------------
GeoDataFrame now has the following CRS:

EPSG:7760


...

/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1129: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.006834113745679666' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, pcen_mobile_colname] = poly_count/denom
/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1212: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.01229964883844355' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, service_idx_colname] = result


transport service index is completed
--------------------------------------------------------
all point services completed
GeoDataFrame now has the following CRS:

EPSG:7760


/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:835: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9072364537233046' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  polygon_gdf.loc[name_index, length_colname] = total_road_length
/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1129: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.003170524126588832' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, pcen_mobile_colname] = poly_count/denom
/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1212: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.01472276507841547' has dtype incompatible with int64, please explicitly cast t

road service is completed


/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1212: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.009351023122807476' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, service_idx_colname] = result


In [45]:
colonies_bbox_psi_popsize.to_csv(colonies_bbox_psi_csv_file)
with open(colonies_bbox_psi_joblib_file, 'wb') as f:
    joblib.dump(colonies_bbox_psi_popsize, f)

colonies_bbox_psi_popsize.head()

,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,area_km2,canal,railway,drain,barrier,...,school_pcen,school_idx,transport_count,transport_pcen,transport_idx,road_length,road_pcen,road_idx,unnorm_psi,norm_psi
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",1.966739,False,True,False,True,...,0.000719,0.001912,3,0.006834,0.012300,1.951928,0.003171,0.014723,0.007666,0.009351
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",0.036429,False,False,False,False,...,0.002299,0.006113,0,0.015705,0.028266,0.000000,0.001126,0.005229,0.030523,0.037231
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",0.230739,False,False,False,False,...,0.003051,0.008112,6,0.004094,0.007369,0.000000,0.000681,0.003162,0.008606,0.010497
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",0.281195,False,False,False,False,...,0.002216,0.005893,0,0.002169,0.003903,0.000000,0.000437,0.002030,0.004520,0.005514
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",0.301253,False,False,False,False,...,0.002003,0.005326,2,0.001473,0.002651,0.747875,0.000650,0.003016,0.002329,0.002841


### Calculate PSI for bbox neighbors using Population Density

In [46]:
colonies_bbox_psi_popdensity = calc_all_services(polygon_gdf = colonies_bbox_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       pcen_denom = "popdensity",
                                       nbr_dist_colname = 'nbrs_dist_bbox')

colonies_bbox_psi_popdensity = colonies_bbox_psi_popdensity.rename(columns={'road_count':'road_length'})

colonies_bbox_psi_popdensity.to_csv(colonies_bbox_psi_popdensity_csv_file)
with open(colonies_bbox_psi_popdensity_joblib_file, 'wb') as f:
    joblib.dump(colonies_bbox_psi_popdensity, f)
    
colonies_bbox_psi_popdensity.head()

GeoDataFrame now has the following CRS:

EPSG:7760


/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1129: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.022553729052259042' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, pcen_mobile_colname] = poly_count/denom
/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1212: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.372781784244884' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, service_idx_colname] = result


bank service index is completed
--------------------------------------------------------
GeoDataFrame now has the following CRS:

EPSG:7760


/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1129: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.00016566929840760984' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, pcen_mobile_colname] = poly_count/denom
/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1212: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.1350192473770908' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, service_idx_colname] = result


health service index is completed
--------------------------------------------------------
GeoDataFrame now has the following CRS:

EPSG:7760


...

transport service index is completed
--------------------------------------------------------
all point services completed
GeoDataFrame now has the following CRS:

EPSG:7760


/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:835: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9072364537233046' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  polygon_gdf.loc[name_index, length_colname] = total_road_length
/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1129: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.006235592778930435' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, pcen_mobile_colname] = poly_count/denom
/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1212: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5892903741369305' has dtype incompatible with int64, please explicitly cast to

road service is completed


/home/bwbelljr/delhi_spatial_index/spatial_index_utils.py:1212: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.4063648733613157' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  gdf_copy.loc[idx, service_idx_colname] = result


,AREA,USO_AREA_U,HOUSETAX_C,USO_FINAL,geometry,area_km2,canal,railway,drain,barrier,...,school_pcen,school_idx,transport_count,transport_pcen,transport_idx,road_length,road_pcen,road_idx,unnorm_psi,norm_psi
0,NEW DELHI 36,5584,None,Planned,"POLYGON Z ((1020282.788 996796.773 0, 1020302....",1.966739,False,True,False,True,...,0.001414,0.188376,3,0.013441,0.411841,1.951928,0.006236,0.589290,0.278927,0.406365
1,NEW DELHI 35,5585,None,Planned,"POLYGON Z ((1019724.475 994932.797 0, 1019788....",0.036429,False,False,False,False,...,0.000084,0.011156,0,0.000572,0.017531,0.000000,0.000041,0.003877,0.035739,0.052067
2,NEW DELHI 34,5586,None,Planned,"POLYGON Z ((1019571.955 994876.019 0, 1019571....",0.230739,False,False,False,False,...,0.000704,0.093782,6,0.000945,0.028948,0.000000,0.000157,0.014846,0.057079,0.083157
3,NEW DELHI 33,5587,None,Planned,"POLYGON Z ((1019352.702 994352.546 0, 1019361....",0.281195,False,False,False,False,...,0.000623,0.083025,0,0.000610,0.018685,0.000000,0.000123,0.011620,0.034187,0.049806
4,NEW DELHI 32,5588,None,Planned,"POLYGON Z ((1018793.292 994224.182 0, 1018573....",0.301253,False,False,False,False,...,0.000603,0.080385,2,0.000444,0.013598,0.747875,0.000196,0.018493,0.019749,0.028773


## Inspecting PSI results

In [47]:
count_cols = [colname for colname in colonies_bbox_psi_popdensity.columns if colname.endswith('_count')]
for count_col in count_cols:
    print('There are', len(colonies_bbox_psi_popdensity[colonies_bbox_psi_popdensity[count_col] < 0]), 'negative values in', count_col, 'column')

There are 0 negative values in bank_count column
There are 0 negative values in health_count column
There are 0 negative values in police_count column
There are 0 negative values in ration_count column
There are 0 negative values in school_count column
There are 0 negative values in transport_count column


In [48]:
pcen_cols = [colname for colname in colonies_bbox_psi_popdensity.columns if colname.endswith('_pcen')]
for pcen_col in pcen_cols:
    print('There are', len(colonies_bbox_psi_popdensity[colonies_bbox_psi_popdensity[pcen_col] < 0]), 'negative values in', pcen_col, 'column')

There are 0 negative values in bank_pcen column
There are 0 negative values in health_pcen column
There are 0 negative values in police_pcen column
There are 0 negative values in ration_pcen column
There are 0 negative values in school_pcen column
There are 0 negative values in transport_pcen column
There are 0 negative values in road_pcen column


In [49]:
idx_cols = [colname for colname in colonies_bbox_psi_popdensity.columns if colname.endswith('_idx')]
for idx_col in idx_cols:
    print('There are', len(colonies_bbox_psi_popdensity[colonies_bbox_psi_popdensity[idx_col] < 0]), 'negative values in', idx_col, 'column')

There are 0 negative values in bank_idx column
There are 0 negative values in health_idx column
There are 0 negative values in police_idx column
There are 0 negative values in ration_idx column
There are 0 negative values in school_idx column
There are 0 negative values in transport_idx column
There are 0 negative values in road_idx column


In [50]:
colonies_bbox_psi_popdensity['bank_idx'].describe()

count    4131.000000
mean        0.012947
std         0.045900
min         0.000000
25%         0.000259
50%         0.002367
75%         0.010612
max         1.000000
Name: bank_idx, dtype: float64

In [51]:
colonies_bbox_psi_popdensity['unnorm_psi'].describe()

count    4131.000000
mean        0.015886
std         0.038548
min         0.000000
25%         0.000762
50%         0.005134
75%         0.017124
max         0.686396
Name: unnorm_psi, dtype: float64

In [52]:
colonies_bbox_psi_popdensity['norm_psi'].describe()

count    4131.000000
mean        0.023144
std         0.056161
min         0.000000
25%         0.001110
50%         0.007480
75%         0.024948
max         1.000000
Name: norm_psi, dtype: float64

## Save Files

In [53]:
bbox_drop_columns = ['nbrs_bbox', 'nbrs_dist_bbox', 'centroid']

In [54]:
colonies_bbox_psi_popsize.drop(columns=bbox_drop_columns).to_file(colonies_bbox_psi_popsize_file)
colonies_bbox_psi_popdensity.drop(columns=bbox_drop_columns).to_file(colonies_bbox_psi_popdensity_file)

/tmp/ipykernel_1080435/477579947.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  colonies_bbox_psi_popsize.drop(columns=bbox_drop_columns).to_file(colonies_bbox_psi_popsize_file)
/home/bwbelljr/anaconda3/envs/delhispatialindex/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'ndmc_dist_km' to 'ndmc_dist_'
  ogr_write(
/home/bwbelljr/anaconda3/envs/delhispatialindex/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'health_count' to 'health_cou'
  ogr_write(
/home/bwbelljr/anaconda3/envs/delhispatialindex/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'health_pcen' to 'health_pce'
  ogr_write(
/home/bwbelljr/anaconda3/envs/delhispatialindex/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'police_count' to 'police_cou'
  ogr_write(
/h

In [55]:
colonies_bbox_psi_popsize.to_csv(colonies_bbox_psi_csv_file)
colonies_bbox_psi_popdensity.to_csv(colonies_bbox_psi_popdensity_csv_file)

In [56]:
with open(colonies_bbox_psi_joblib_file, 'wb') as f:
    joblib.dump(colonies_bbox_psi_popsize, f)
    
with open(colonies_bbox_psi_popdensity_joblib_file, 'wb') as f:
    joblib.dump(colonies_bbox_psi_popdensity, f)